In [1]:
from compilers import (
    PipelineGenerator,
    PipelineValidator,
    FileMaterializer,
    LdioConfigCompiler,
    ValidationReportCompiler,
)

#### Validating the pipeline definition

`PipelineValidator(pipeline_id)` returns a `CompilationRunner` wrapped around `PipelineValidatorConfig` — the shared preparation compilers (`PipelineSeeder`, `PipelineAssembler`, `PipelineEnricher`, `BridgeTransportCompiler`, `SegmentTagger`, `GraphReducer`, and the per-boundary config compilers) plus `ValidationReportCompiler` gated on the finalize phase. `sh:conforms == True` means the pipeline definition passes SHACL against the application profile and is safe to hand to `PipelineGenerator`.

In [ ]:
val = PipelineValidator(pipeline_id := "demo:DishacledPipeline")
val.compile()
val.compilers[ValidationReportCompiler].conforms

#### Compiling the pipeline build

`PipelineGenerator(pipeline_id)` returns a `CompilationRunner` wrapped around `PipelineGeneratorConfig`. The runner loads every ttl in the config's `graph_files` (catalog + shipped pipeline definitions) into one rdflib graph, applies the inference rules, then attaches a `tcs:CompilationRequest` node carrying `tcs:targetPipeline <pipeline_id>`. `PipelineSeeder` fires first because its `applies_to` trigger looks for that request; the remaining compilers fall into place as each preceding one adds the triples the next one needs. Once the loop settles, the runner attaches `<request> tcs:runPhase tcs:FinalizePhase` and re-runs the fixpoint; `ValidationReportCompiler` and `DockerComposeCompiler` gate on that marker, so they only fire in this finalize pass.

In [2]:
pipeline_id = "demo_sd:SemanticsDemoPipeline"
gen = PipelineGenerator(pipeline_id)
build_graph = gen.compile()

# Which compilers actually ran?
[cls.__name__ for cls in gen.compilers]

['PipelineSeeder',
 'SemanticModelMapper',
 'PipelineAssembler',
 'PipelineEnricher',
 'BridgeTransportCompiler',
 'SegmentTagger',
 'GraphReducer',
 'RdfcHttpServerConfigCompiler',
 'ConfigTranslator',
 'RdfcDockerFileCompiler',
 'LdioHttpOutConfigCompiler',
 'RdfcConfigCompiler',
 'LdioConfigCompiler',
 'ContainerServiceNameCompiler',
 'ValidationReportCompiler',
 'DockerComposeCompiler']

#### Inspecting the compiled files

Every compiler that produces a file attaches it to the `tcs:PipelineBuild` as an `spdx:File` node via `tcs:compiledFile`. The build graph is now self-describing: it knows which files should be written, where, and with what content. `FileMaterializer` collects those nodes into a DataFrame on `builder.files` for inspection before any IO happens.

In [4]:
builder = FileMaterializer(build_graph)

for _, row in builder.files.iterrows():
    print(f"=== {row['filepath']}/{row['filename']} ===")
    print(row['content'])
    print()

=== validation/validation-report.ttl ===
@base <file:///workspace/pipeline/> .
@prefix : <http://example.org/example/> .
@prefix ldio: <http://example.org/example/ldio/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix tcs: <https://w3id.org/toolchain#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:ChangeSemanticsShape tcs:passed true .

:HttpFetchAuthShape tcs:passed true .

:HttpFetchOptionsShape tcs:passed true .

:HttpServerOptionsShape tcs:passed true .

:HttpServerShape tcs:passed true .

:IngestConfigShape tcs:passed true .

:LogProcessorJsShape tcs:passed true .

:MetadataConfigShape tcs:passed true .

:PerformanceConfigShape tcs:passed true .

:TransactionConfigShape tcs:passed true .

ldio:HttpInPollerConfigShape sh:message "Checks that ldio:HttpInPoller's wiring (url/cron/interval/continueOnFail plus shared HTTP-requester options) is well-typed." ;
    tcs:passed true .

ldio:HttpOutConfigShape sh:message "Checks that ldio:HttpOut's wiring (endpoint/rdf-writer 

#### Writing the project to disk

`FileMaterializer.write(target_dir)` materializes every collected file under the given directory, creating parent folders as needed. Existing files at the same path are overwritten. The call returns the absolute paths it wrote.

In [ ]:
from pathlib import Path

full_out_dir = Path("../out/dishacled-full").resolve()
written = builder.write(str(full_out_dir))
for path in written:
    print(path)

C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\rdfc\Dockerfile
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\error-alert\config.json
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\virtuoso\virtuoso.ini
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\validation\validation-report.ttl
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\rdfc\pyproject.toml
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\rdfc\pipeline.ttl
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-

#### Inspecting compiler internals

`PipelineGenerator` keeps the compiler instances it ran on `gen.compilers`, keyed by class. Each instance retains its intermediate state — useful for debugging when an output doesn't look right.

In [ ]:
gen.compilers[LdioConfigCompiler].df_steps

,component,type,name,config
0,ldio:HttpInPoller,Input,Ldio:HttpInPoller,:config_19
1,ldio:JsonToLdAdapter,Adapter,Ldio:JsonToLdAdapter,:config_6
2,ldio:SparqlConstructTransformer,Transformer,Ldio:SparqlConstructTransformer,:config_58
3,ldio:HttpOut,Output,Ldio:HttpOut,:config_34


#### Inspecting what each compiler added and removed

Every compiler on `gen.compilers` exposes `.added_triples` and `.removed_triples` — `GraphReader` views over the delta between its `input_reader` (snapshot at construction time) and its `output_reader` (final state after `compile()`). Together they make the compilation process fully transparent: for any compiler, you can see exactly which triples it contributed and which it removed.

`SemanticWorksEnvVarCompiler` is a good example because it does both: it strips the old `tcs:literal` (or `tcs:embedded`) body of each SemanticWorks `tcs:DockerComposeConfig` and writes back an updated one with the step's config folded into the service `environment`.

In [ ]:
from compilers import SemanticWorksEnvVarCompiler

sw = gen.compilers[SemanticWorksEnvVarCompiler]

print(f"Triples added by SemanticWorksEnvVarCompiler: {len(sw.added_triples.df)}")
print(f"Triples removed by SemanticWorksEnvVarCompiler: {len(sw.removed_triples.df)}")

print("\n--- added ---")
display(sw.added_triples.df)
print("\n--- removed ---")
display(sw.removed_triples.df)

Triples added by SemanticWorksEnvVarCompiler: 2
Triples removed by SemanticWorksEnvVarCompiler: 2

--- added ---


,sub,pred,obj,sub_type,obj_type
0,:LoketErrorAlertServiceDockerCompose,tcs:literal,"{""services"": {""error-alert"": {""image"": ""lblod/...",<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
1,:DeliverEmailServiceDockerCompose,tcs:literal,"{""services"": {""berichtencentrum-deliver-email-...",<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>



--- removed ---


,sub,pred,obj,sub_type,obj_type
0,:LoketErrorAlertServiceDockerCompose,tcs:literal,\r\nerror-alert:\r\n image: lblod/loket-err...,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
1,:DeliverEmailServiceDockerCompose,tcs:literal,\r\n berichtencentrum-deliver-email-service:\...,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
